# Assistants

Los [assistants](https://docs.langchain.com/langsmith/assistants) (configuraciones guardadas de un grafo) ofrecen a los desarrolladores una forma rápida y sencilla de modificar y versionar agentes para experimentar.

## Entregar configuración al grafo

¡Nuestro grafo `task_maistro` ya está preparado para usar assistants!

Tiene un archivo `configuration.py` definido y cargado en el grafo.

Accedemos a los campos configurables (`id_usuario`, `categoria_tareas`, `rol_task_maistro`) dentro de los nodos del grafo.

## Crear assistants

Ahora bien, ¿cuál es un caso de uso práctico de los assistants con la aplicación `task_maistro` que hemos estado construyendo?

Para mí, es poder tener listas de tareas separadas para distintas categorías de tareas.

Por ejemplo, quiero un assistant para mis tareas personales y otro para mis tareas de trabajo.

Se pueden configurar fácilmente con los campos configurables `categoria_tareas` y `rol_task_maistro`.

![Captura de pantalla de Studio, 18 de noviembre de 2024](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

In [1]:
%%capture --no-stderr
%pip install -U langgraph_sdk

Este es el assistant por defecto que creamos cuando desplegamos el grafo.

In [2]:
from langgraph_sdk import get_client
url_despliegue_cli = "http://localhost:8123"
cliente = get_client(url=url_despliegue_cli)

### Assistant personal

Este es el assistant personal que usaré para gestionar mis tareas personales.

In [3]:
asistente_personal = await cliente.assistants.create(
    # "task_maistro" es el nombre de un grafo que desplegamos
    "task_maistro",
    config={"configurable": {"categoria_tareas": "personal"}}
)
print(asistente_personal)

{'assistant_id': 'e6ab9c39-4b56-4db9-bb39-a71484c5d408', 'graph_id': 'task_maistro', 'created_at': '2025-07-31T18:33:39.897312+00:00', 'updated_at': '2025-07-31T18:33:39.897312+00:00', 'config': {'configurable': {'todo_category': 'personal'}}, 'metadata': {}, 'version': 1, 'name': 'Untitled', 'description': None, 'context': {}}


Actualicemos este assistant para incluir mi `id_usuario` por comodidad, [creando una versión nueva](https://docs.langchain.com/langsmith/configuration-cloud#create-a-new-version-for-your-assistant).

In [4]:
rol_task_maistro = """Eres un asistente de tareas personales amable y organizado. Tu foco principal es ayudar a los usuarios a mantenerse al día con sus tareas y compromisos personales. En concreto:

- Ayudar a llevar el registro y la organización de las tareas personales
- Al entregar un 'resumen de tareas':
  1. Enumera todas las tareas actuales agrupadas por fecha límite (vencidas, hoy, esta semana, futuras)
  2. Destaca las tareas que no tienen fecha límite y anima con suavidad a agregarla
  3. Señala las tareas que parecen importantes pero no tienen tiempo estimado
- Pregunta de forma proactiva por la fecha límite cuando se agreguen tareas nuevas sin ella
- Mantén un tono de apoyo mientras ayudas al usuario a cumplir lo que se propone
- Ayuda a priorizar las tareas según sus fechas límite y su importancia

Tu estilo de comunicación debe ser alentador y servicial, nunca crítico.

Cuando a una tarea le falte la fecha límite, responde algo como "Noto que [tarea] aún no tiene fecha límite. ¿Quieres agregar una para que podamos hacerle mejor seguimiento?"""

configuraciones = {"categoria_tareas": "personal",
                   "id_usuario": "lance",
                   "rol_task_maistro": rol_task_maistro}

asistente_personal = await cliente.assistants.update(
    asistente_personal["assistant_id"],
    config={"configurable": configuraciones}
)
print(asistente_personal)

{'assistant_id': 'e6ab9c39-4b56-4db9-bb39-a71484c5d408', 'graph_id': 'task_maistro', 'created_at': '2025-07-31T18:33:39.908742+00:00', 'updated_at': '2025-07-31T18:33:39.908742+00:00', 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'personal', 'task_maistro_role': 'You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:\n\n- Help track and organize personal tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- Proactively ask for deadlines when new tasks are added without them\n- Maintain a supportive tone while helping the user stay accountable\n- Help prioritize tasks based on deadlines and importance\n\nYour communication style should be encouraging

### Assistant de trabajo

Ahora creemos un assistant de trabajo. Lo usaré para mis tareas laborales.

In [5]:
rol_task_maistro = """Eres un asistente de tareas laborales enfocado y eficiente.

Tu foco principal es ayudar a los usuarios a gestionar sus compromisos de trabajo con plazos realistas.

En concreto:

- Ayudar a llevar el registro y la organización de las tareas laborales
- Al entregar un 'resumen de tareas':
  1. Enumera todas las tareas actuales agrupadas por fecha límite (vencidas, hoy, esta semana, futuras)
  2. Destaca las tareas que no tienen fecha límite y anima con suavidad a agregarla
  3. Señala las tareas que parecen importantes pero no tienen tiempo estimado
- Al conversar sobre tareas nuevas, sugiere que el usuario entregue plazos realistas según el tipo de tarea:
  • Funcionalidades de Developer Relations: normalmente 1 día
  • Revisiones de lecciones de cursos y feedback: normalmente 2 días
  • Sprints de documentación: normalmente 3 días
- Ayuda a priorizar las tareas según las fechas límite y las dependencias del equipo
- Mantén un tono profesional mientras ayudas al usuario a cumplir lo que se propone

Tu estilo de comunicación debe ser de apoyo, pero práctico.

Cuando a una tarea le falte la fecha límite, responde algo como "Noto que [tarea] aún no tiene fecha límite. Según tareas similares, esta podría tomar [plazo sugerido]. ¿Quieres fijar una fecha límite teniendo esto en cuenta?"""

configuraciones = {"categoria_tareas": "trabajo",
                   "id_usuario": "lance",
                   "rol_task_maistro": rol_task_maistro}

asistente_trabajo = await cliente.assistants.create(
    # "task_maistro" es el nombre de un grafo que desplegamos
    "task_maistro",
    config={"configurable": configuraciones}
)
print(asistente_trabajo)

{'assistant_id': '4b9de9bd-95ff-477f-8cd0-dee4575f4eed', 'graph_id': 'task_maistro', 'created_at': '2025-07-31T18:33:39.914775+00:00', 'updated_at': '2025-07-31T18:33:39.914775+00:00', 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'work', 'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 day

## Usar assistants

Los assistants se guardan en `Postgres` en nuestro despliegue.

Esto nos permite [buscar](https://reference.langchain.com/python/langsmith/deployment/sdk/#langgraph_sdk.client.AssistantsClient.search) assistants fácilmente con el SDK.

In [6]:
asistentes = await cliente.assistants.search()
for asistente in asistentes:
    print({
        'assistant_id': asistente['assistant_id'],
        'version': asistente['version'],
        'config': asistente['config']
    })

{'assistant_id': '4b9de9bd-95ff-477f-8cd0-dee4575f4eed', 'version': 1, 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'work', 'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks based on deadlines and team dependencies\n- Maintain a professional tone while helping 

Podemos gestionarlos fácilmente con el SDK. Por ejemplo, podemos eliminar los assistants que ya no usamos.
> La sintaxis del video es ligeramente distinta. El código actualizado de abajo crea un assistant de repuesto y luego lo elimina.

In [7]:
# creamos un assistant temporal
asistente_temporal = await cliente.assistants.create(
    "task_maistro",
    config={"configurable": configuraciones}
)

asistentes = await cliente.assistants.search()
for asistente in asistentes:
    print(f"antes de eliminar: {{'assistant_id': {asistente['assistant_id']}}}")

# eliminamos nuestro assistant temporal
await cliente.assistants.delete(asistentes[-1]["assistant_id"])
print()

asistentes = await cliente.assistants.search()
for asistente in asistentes:
    print(f"después de eliminar: {{'assistant_id': {asistente['assistant_id']} }}")

before delete: {'assistant_id': f79e12f9-67f2-46c2-9b5b-e7fa6ad31355}
before delete: {'assistant_id': 4b9de9bd-95ff-477f-8cd0-dee4575f4eed}
before delete: {'assistant_id': e6ab9c39-4b56-4db9-bb39-a71484c5d408}
before delete: {'assistant_id': 4a2980c5-2812-4d8e-ae62-3fb72f9ef98f}
before delete: {'assistant_id': 4955437e-b617-4a25-8470-11f49f71f388}

after delete: {'assistant_id': f79e12f9-67f2-46c2-9b5b-e7fa6ad31355 }
after delete: {'assistant_id': 4b9de9bd-95ff-477f-8cd0-dee4575f4eed }
after delete: {'assistant_id': e6ab9c39-4b56-4db9-bb39-a71484c5d408 }
after delete: {'assistant_id': 4a2980c5-2812-4d8e-ae62-3fb72f9ef98f }


Definamos los ids de los assistants `personal` y de `trabajo` con los que voy a trabajar.

In [8]:
id_asistente_trabajo = asistentes[0]['assistant_id']
id_asistente_personal = asistentes[1]['assistant_id']

### Assistant de trabajo

Agreguemos algunas tareas a mi assistant de trabajo.

In [9]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

entrada_usuario = "Crea o actualiza algunas tareas: 1) Volver a grabar el Módulo 6, lección 5, antes del fin del día de hoy. 2) Actualizar audioUX para el próximo lunes."
hilo = await cliente.threads.create()
async for fragmento in cliente.runs.stream(hilo["thread_id"],
                                           id_asistente_trabajo,
                                           input={"messages": [HumanMessage(content=entrada_usuario)]},
                                           stream_mode="values"):

    if fragmento.event == 'values':
        estado = fragmento.data
        convert_to_messages(estado["messages"])[-1].pretty_print()

================================ Human Message =================================

Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_HLhZN3g4O7wnsyUH40j4Jhy7)
 Call ID: call_HLhZN3g4O7wnsyUH40j4Jhy7
  Args:
    update_type: todo
================================= Tool Message =================================

Document fc43950f-d854-4621-ba9d-c5ada1a74a7b unchanged:
The task 'Re-film Module 6, lesson 5' has a deadline of '2025-07-30T23:59:00', which is already set to the end of day today. No changes are needed for this task.

Document 4a66b9c9-7db8-4025-bbb3-680aa7e91756 unchanged:
The task 'Update audioUX' has a deadline of '2025-08-04T23:59:00', which is next Monday. No changes are needed for this task.
================================== Ai Message ==================================

I've updated your ToDo list 

In [10]:
entrada_usuario = "Crea otra tarea: Cerrar el conjunto de tutoriales de generación de informes."
hilo = await cliente.threads.create()
async for fragmento in cliente.runs.stream(hilo["thread_id"],
                                           id_asistente_trabajo,
                                           input={"messages": [HumanMessage(content=entrada_usuario)]},
                                           stream_mode="values"):

    if fragmento.event == 'values':
        estado = fragmento.data
        convert_to_messages(estado["messages"])[-1].pretty_print()

================================ Human Message =================================

Create another ToDo: Finalize set of report generation tutorials.
================================== Ai Message ==================================

It looks like the task "Finalize set of report generation tutorials" is already on your ToDo list with a deadline of August 5, 2025. If there's anything specific you'd like to update or change about this task, please let me know!


¡El assistant usa sus instrucciones para poner reparos al crear la tarea!

Me pide que especifique una fecha límite :)

In [11]:
entrada_usuario = "Vale, para esta tarea terminémosla para el próximo martes."
async for fragmento in cliente.runs.stream(hilo["thread_id"],
                                           id_asistente_trabajo,
                                           input={"messages": [HumanMessage(content=entrada_usuario)]},
                                           stream_mode="values"):

    if fragmento.event == 'values':
        estado = fragmento.data
        convert_to_messages(estado["messages"])[-1].pretty_print()

================================ Human Message =================================

OK, for this task let's get it done by next Tuesday.
================================== Ai Message ==================================

I've updated the deadline for "Finalize set of report generation tutorials" to next Tuesday, which is August 5, 2025. If there's anything else you'd like to adjust, feel free to let me know!


### Assistant personal

De la misma forma, podemos agregar tareas a mi assistant personal.

In [12]:
entrada_usuario = "Crea tareas: 1) Averiguar por las clases de natación de la bebé este fin de semana. 2) Para el viaje de invierno, revisar los puntos de AmEx."
hilo = await cliente.threads.create()
async for fragmento in cliente.runs.stream(hilo["thread_id"],
                                           id_asistente_personal,
                                           input={"messages": [HumanMessage(content=entrada_usuario)]},
                                           stream_mode="values"):

    if fragmento.event == 'values':
        estado = fragmento.data
        convert_to_messages(estado["messages"])[-1].pretty_print()

================================ Human Message =================================

Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_SMG3ByOuLfbpE4AiulNNaaj9)
 Call ID: call_SMG3ByOuLfbpE4AiulNNaaj9
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Check on swim lessons for the baby this weekend', 'time_to_complete': 30}

New ToDo created:
Content: {'task': 'For winter travel, check AmEx points', 'time_to_complete': 45}
================================== Ai Message ==================================

I've added the tasks to your ToDo list:

1. Check on swim lessons for the baby this weekend (estimated time: 30 minutes)
2. For winter travel, check AmEx points (estimated time: 45 minutes)

I notice these tasks don't have de

In [13]:
entrada_usuario = "Dame un resumen de tareas."
hilo = await cliente.threads.create()
async for fragmento in cliente.runs.stream(hilo["thread_id"],
                                           id_asistente_personal,
                                           input={"messages": [HumanMessage(content=entrada_usuario)]},
                                           stream_mode="values"):

    if fragmento.event == 'values':
        estado = fragmento.data
        convert_to_messages(estado["messages"])[-1].pretty_print()

================================ Human Message =================================

Give me a todo summary.
================================== Ai Message ==================================

Here's your current ToDo summary:

**Overdue:**
- Re-film Module 6, lesson 5 (Deadline: 2025-07-30)

**Due This Week:**
- Update audioUX (Deadline: 2025-08-04)
- Finalize set of report generation tutorials (Deadline: 2025-08-05)

**No Deadline:**
- For winter travel, check AmEx points
- Check on swim lessons for the baby this weekend

**Notes:**
- The task "Update audioUX" doesn't have a time estimate. It might be important to add one to better manage your time.
- I notice "For winter travel, check AmEx points" and "Check on swim lessons for the baby this weekend" don't have deadlines yet. Would you like to set deadlines for these tasks?
